In [1]:
import introns
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
import matplotlib.pyplot as plt
from Bio import SeqIO
from copy import copy
from collections import defaultdict

# Checking manual loading

In [2]:
genes, genome = introns.create_manual_from_gb('./fastas_and_gff/Alignments/alignments_all_clean_3_genes_only.gb')
#genes, genome = introns.create_manual_from_gb('./fastas_and_gff/Alignments/alignments_all_clean_3.gb')

predict_all_introns checkpoint 1/4
predict_all_introns checkpoint 2/4
predict_all_introns checkpoint 3/4
predict_all_introns checkpoint 4/4


Wlasne proby napisania parsera

In [3]:
f = open('./fastas_and_gff/Alignments/alignments_all_clean_3_genes_only.gb', encoding='utf-8')
records = SeqIO.parse(f, 'genbank')

genes = {}
genome = defaultdict(str)
intron_labels = introns.create_label_dict()

In [4]:
record = next(records)
gene_name, gene = record.name, None
list_of_exons = []
list_to_make_introns = []
gene_start, gene_end = 1000000, 0
sequence = str(record.seq)
genome[gene_name] = sequence

if record.features[0].type == 'intron':
    del record.features[0]
if record.features[-1].type == 'intron':
    del record.features[-1]
    



for feature in record.features:
    feature_start, feature_end = int(feature.location.start), int(feature.location.end)
    if feature.type == 'exon':
        if feature_start < gene_start:
            gene_start = feature_start
        if feature_end > gene_end:
            gene_end = feature_end
        exon = introns.Exon(gene_name, feature_start, feature_end, strand='+', gene=None)
        list_of_exons.append(exon)
    elif feature.type == 'misc_feature':
        ft_name = feature.qualifiers.get('standard_name')[0]
        if ft_name in intron_labels:
            #intron = introns.Intron(gene_name, feature_start, feature_end, strand='+',
            #                        gene=None, man_annotation=intron_labels[ft_name])
            #list_to_make_introns.append(intron)
            list_to_make_introns.append(
                (gene_name, feature_start, feature_end, intron_labels[ft_name])
                )
        
gene = introns.Gene(gene_name, gene_start, gene_end, name=gene_name, strand='+',
                    exons=[], working_exons = list_of_exons)
#list_of_introns = list_to_make_introns
list_of_introns = []
for (gene_name, start, end, label) in list_to_make_introns:
    intron = introns.Intron(gene_name, start, end, strand='+',
                            gene=gene, man_annotation=label)
    list_of_introns.append(intron)

gene.working_introns = list_of_introns

tu jest robienie exonow

In [5]:
for exon in gene.working_exons:
    assert gene.scaffold_start <= exon.scaffold_start < exon.scaffold_end <= gene.scaffold_end
    print(exon.scaffold_start, exon.scaffold_end)

504 615
4197 4262
4347 4390
6768 6883
29032 29163
29473 29509
29950 29998
30838 30925
30925 30948
30948 30997
31362 31472
31686 31762
31762 31865
36193 36231
36231 36255
36386 36431
36705 36828
37717 37862


In [6]:
genes_to_iter = gene.working_exons+[None]
loop = list(enumerate(genes_to_iter))
loop_gen = iter(loop)

ready_exons = []

for i, exon in loop_gen:
    next_exon = genes_to_iter[i+1]
    if not next_exon:
        ready_exons.append(exon)
        break
    
    print(i, exon.scaffold_end == next_exon.scaffold_start)
    
    if exon.scaffold_end == next_exon.scaffold_start:
        try:
            i_skipped, exon_skipped = next(loop_gen)
        except IndexError:
            print(f"\tlast two introns {i_skipped, exon_skipped}")
        assert next_exon==exon_skipped
        print(f"\tmodifying exon {i_skipped, exon_skipped}")
        
        new_exon = copy(exon)
        new_exon.sequence = exon.sequence + next_exon.sequence
        new_exon.scaffold_end = next_exon.scaffold_end
        new_exon.next_exon = next_exon.next_exon
        
        ready_exons.append(new_exon)
    
    else:
        ready_exons.append(exon)
        

0 False
1 False
2 False
3 False
4 False
5 False
6 False
7 True
	modifying exon (8, EL|STRG.5113.1|scaffold_3161:4719-42082 30925 30948)
9 False
10 False
11 True
	modifying exon (12, EL|STRG.5113.1|scaffold_3161:4719-42082 31762 31865)
13 True
	modifying exon (14, EL|STRG.5113.1|scaffold_3161:4719-42082 36231 36255)
15 False
16 False


In [7]:
def iterate_list(l):
    for i, item in enumerate(l):
        prev = None if i==0 else l[i-1]
        next = None if i==len(l)-1 else l[i+1]
        yield prev, item, next

for prev_exon, curr_exon, next_exon in iterate_list(ready_exons):
    curr_exon.prev_exon = prev_exon
    curr_exon.next_exon = next_exon

gene.exons = ready_exons

Tu sie dodaje introny

In [8]:
for intron in gene.working_introns:
    assert gene.scaffold_start <= intron.scaffold_start < intron.scaffold_end <= gene.scaffold_end
    print(intron.scaffold_start, intron.scaffold_end)

615 4197
4262 4347
4390 6768
6883 29032
29163 29473
29509 29950
29998 30838
30997 31362
31472 31686
31865 36193
36255 36386
36431 36705
36828 37717


In [9]:
print(len(gene.exons), len(gene.working_introns))
print(gene.working_introns[-1])
print(gene.exons[-1])

15 13

    Intron in gene: EL|STRG.5113.1|scaffold_3161:4719-42082
    scaff loc: 36828-37717
    gene loc: 36324-37213

    Exon in gene: EL|STRG.5113.1|scaffold_3161:4719-42082
    scaff loc: 37717-37862


In [10]:
i_g = (n for n in gene.working_introns)
e_g = (n for n in gene.exons)
while True:
    try:
        i, e = next(i_g), next(e_g)
    except StopIteration:
        break
    print(f"{e.scaffold_start}:{e.scaffold_end}, {i.scaffold_start}:{i.scaffold_end}")

504:615, 615:4197
4197:4262, 4262:4347
4347:4390, 4390:6768
6768:6883, 6883:29032
29032:29163, 29163:29473
29473:29509, 29509:29950
29950:29998, 29998:30838
30838:30948, 30997:31362
30948:30997, 31472:31686
31362:31472, 31865:36193
31686:31865, 36255:36386
36193:36255, 36431:36705
36386:36431, 36828:37717


In [11]:
f.close()